# Part 4c — Cournot competition with endogenous price

### Where flooding the market becomes rational

Part 4b had a **fixed** price, so rivalry was a pure race for a capped market and the only channel
between firms was residual demand. That produced two artefacts: a large first-mover advantage, and
an equilibrium that served less demand than a planner would.

Here price responds to quantity:

$$p_{rt,p} \;=\; A_{rt,p} - B_{rt,p} \sum_{r} \text{sale}_{r,rt,p}$$

That single change alters the economics qualitatively. Extra output now **depresses the rival's
margin as well as your own** — and because operating-cost learning is driven by cumulative
production, extra output also **advances your own learning tier**. Those two effects together are
the documented predatory dynamic: flood the market, starve the entrant, and come out cheaper.

### What is new

| | 4b (fixed price) | **4c (Cournot)** |
|---|---|---|
| Price | exogenous constant | $A - BQ$, endogenous |
| Firm objective | $\bar{p}\cdot\text{sales} - \text{cost}$ | quadratic revenue − cost |
| Rivalry channel | residual demand cap | **the price itself** |
| Quantity cap | rival's leftovers | choke price only |
| Benchmark | cooperative planner | **collusion** (joint profit max) |

## The formulation problem, and a neat way around it

Firm $r$'s revenue in market $rt$, taking the rival's quantity $\bar{q}$ as given:

$$\big(A - B(s + \bar{q})\big)\, s \;=\; \underbrace{(A - B\bar{q})}_{a_{\text{eff}}}\, s \;-\; B s^2$$

This is **quadratic**, so each best response is naturally a MIQP. Two reasons not to do that:
Gurobi's size-limited `pip` licence does not accept quadratic objectives at this model size, and a
MIQP is heavier than it needs to be.

**Piecewise-linearise the revenue instead.** And here the curvature works in our favour:

| | Minimising | Maximising |
|---|---|---|
| **Convex** | safe | chord exploited |
| **Concave** | **chord exploited** (Part 3's learning curve) | **safe** ← we are here |

Revenue is **concave** and we **maximise**, so every chord lies *below* the true curve. A free
convex combination of breakpoints has no incentive to mix non-adjacent points — mixing would report
*less* revenue. So this needs **no SOS2 and adds no binaries**, unlike Part 3's cumulative cost
curve. It is the exact mirror image, and worth pausing on: whether you need SOS2 depends on
curvature *and* optimisation direction together, never on one alone.

## 1. Setup and the shared model core

Everything is inherited from Part 4a/4b: the same `add_region` chain, the asymmetric instance
(R1 incumbent with 2,600 units of accumulated experience, R2 cheaper entrant with 500), both
learning channels, variable-length periods.

In [ ]:
!pip install gurobipy --quiet
import math
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})
print("gurobipy", gp.gurobi.version())

In [ ]:
REGIONS = ['R1', 'R2']
STAGES  = ['MINE', 'PROC', 'MFG']

# ---------------- TIME ----------------
BLOCKS = [(6, 1), (4, 3), (2, 5), (1, 9)]
LEN, START = [], []
_y = 1
for _c, _L in BLOCKS:
    for _ in range(_c):
        LEN.append(_L); START.append(_y); _y += _L
P = list(range(len(LEN)))
HORIZON = _y - 1
YEARS = {p: list(range(START[p], START[p] + LEN[p])) for p in P}
DR = 0.05
OMEGA = {p: sum(1/(1+DR)**t for t in YEARS[p]) for p in P}
YEAR_TO_P = {t: p for p in P for t in YEARS[p]}
REPORT_UNTIL = 28

# ---------------- TECH ----------------
LIFE = 25
LEAD = {'MINE': 1, 'PROC': 2, 'MFG': 2}
CAP_MIN, CAP_MAX = 60.0, 260.0
CRF = DR*(1+DR)**LIFE/((1+DR)**LIFE - 1)
ONLINE = {(s, p): START[p] + LEAD[s] for s in STAGES for p in P}
MU = {(s, v): CRF*sum(1/(1+DR)**t for t in range(ONLINE[s, v], ONLINE[s, v]+LIFE)
                      if t <= HORIZON) for s in STAGES for v in P}

In [ ]:
# ---------------- ASYMMETRIC INSTANCE ----------------
# R1 = incumbent upstream processor with accumulated experience.
# R2 = entrant, cheaper to build, trying to move downstream.
FIXED = {('MINE','R1'):900.,('PROC','R1'):1500.,('MFG','R1'):1300.,
         ('MINE','R2'):820.,('PROC','R2'):1350.,('MFG','R2'):1180.}
UNIT  = {('MINE','R1'):7.0,('PROC','R1'):11.0,('MFG','R1'):9.5,
         ('MINE','R2'):6.4,('PROC','R2'):10.0,('MFG','R2'):8.7}
OPEX  = {('MINE','R1'):1.2,('PROC','R1'):2.0,('MFG','R1'):2.4,
         ('MINE','R2'):1.35,('PROC','R2'):2.2,('MFG','R2'):2.6}

LEGACY_CAP = {('MINE','R1'):230,('PROC','R1'):205,('MFG','R1'):155,
              ('MINE','R2'):165,('PROC','R2'):125,('MFG','R2'):100}
LEGACY_RET = {('MINE','R1'):11,('PROC','R1'):14,('MFG','R1'):18,
              ('MINE','R2'):9, ('PROC','R2'):16,('MFG','R2'):22}
LEGACY_BYR = -8
# incumbent starts with accumulated production experience
EXPERIENCE0 = {'R1': 2600.0, 'R2': 500.0}

In [ ]:
# ---------------- EFFICIENCY (yield) ----------------
ETA_CEIL = {'MINE':0.92,'PROC':0.95,'MFG':0.93}
ETA_BASE = {'MINE':0.86,'PROC':0.80,'MFG':0.78}
ALPHA    = {'MINE':0.0,'PROC':0.030,'MFG':0.025}
BETA     = {'MINE':0.0,'PROC':0.010,'MFG':0.008}
DELTA_BAR= {'MINE':0.02,'PROC':0.05,'MFG':0.05}
ETA_FLOOR= 0.60
VINTAGES = [-1] + P
BYEAR = {v: (LEGACY_BYR if v == -1 else START[v]) for v in VINTAGES}
ETA = {}
for s in STAGES:
    for v in VINTAGES:
        fr = ETA_CEIL[s] - (ETA_CEIL[s]-ETA_BASE[s])*(1-ALPHA[s])**(BYEAR[v]-1)
        fr = max(ETA_FLOOR, min(fr, ETA_CEIL[s]))
        for p in P:
            age = max(0, START[p]-BYEAR[v])
            aged = ETA_CEIL[s] - (ETA_CEIL[s]-fr)*(1-BETA[s])**age
            ETA[s, v, p] = max(ETA_FLOOR, min(fr+DELTA_BAR[s], aged))

In [ ]:
# ---------------- DEMAND & MARKET ----------------
DEMAND = {}
for r, base, g in [('R1', 100.0, 0.008), ('R2', 75.0, 0.026)]:
    for p in P:
        DEMAND[r, p] = sum(base*(1+g)**(t-1) for t in YEARS[p])/LEN[p]
TRANSPORT = {(rf, rt): (0.5 if rf == rt else 2.4) for rf in REGIONS for rt in REGIONS}
PRICE_FIXED = 12.0
PEN_SHORT, PEN_DISPOSE = 90.0, 12.0

In [ ]:
# ---------------- LEARNING ----------------
LEARN_STAGES = ['PROC', 'MFG']
LR_CAPEX, Q_START, Q_ADD, CAPEX_FLOOR, NBP = 0.15, 300.0, 700.0, 0.60, 9
_bc = -math.log2(1-LR_CAPEX)
K = list(range(NBP))
QBP = [Q_START + Q_ADD*k/(NBP-1) for k in K]

def _cap_unit_mult(q):
    return max(CAPEX_FLOOR, (q/Q_START)**(-_bc))

def _cap_cum_mult(q, n=400):
    if q <= Q_START:
        return 0.0
    h = (q-Q_START)/n
    return sum(0.5*(_cap_unit_mult(Q_START+i*h)+_cap_unit_mult(Q_START+(i+1)*h))*h
               for i in range(n))
CBP = [_cap_cum_mult(q) for q in QBP]

LR_OPEX, OPEX_FLOOR, LAG_YEARS, N_TIERS = 0.18, 0.65, 3, 3
TIER_Q, TIER_M = {}, {}

def set_tiers(top_by_region):
    for r in REGIONS:
        top = max(top_by_region[r], 1.0)
        q1 = top/8.0
        TIER_Q[r] = [q1*2**j for j in range(N_TIERS-1)]
        TIER_M[r] = [max(OPEX_FLOOR, (1-LR_OPEX)**j) for j in range(N_TIERS)]

ACTIVE = {r: [(s, v, p) for s in STAGES for v in VINTAGES for p in P
              if (v == -1 and START[p] <= LEGACY_RET[s, r])
              or (v >= 0 and ONLINE[s, v] <= START[p] <= ONLINE[s, v]+LIFE-1)]
          for r in REGIONS}
VIN = {(r, s, p): [v for (ss, v, pp) in ACTIVE[r] if (ss, pp) == (s, p)]
       for r in REGIONS for s in STAGES for p in P}
BUILD = {r: [(s, v) for s in STAGES for v in P if ONLINE[s, v] <= HORIZON]
         for r in REGIONS}

In [ ]:
def add_region(m, r, learning='both'):
    """Attach one region's vertically-integrated chain to model m. Returns handles."""
    b = m.addVars(BUILD[r], vtype=GRB.BINARY, name=f'b_{r}')
    c = m.addVars(BUILD[r], lb=0.0, ub=CAP_MAX, name=f'c_{r}')
    x = m.addVars(ACTIVE[r], lb=0.0, name=f'x_{r}')
    f_mp = m.addVars(P, lb=0.0, name=f'fmp_{r}')
    f_pf = m.addVars(P, lb=0.0, name=f'fpf_{r}')
    sale = m.addVars(REGIONS, P, lb=0.0, name=f'sale_{r}')
    disp = m.addVars(P, lb=0.0, name=f'disp_{r}')

    m.addConstrs((c[s, v] <= CAP_MAX*b[s, v] for (s, v) in BUILD[r]), name=f'su_{r}')
    m.addConstrs((c[s, v] >= CAP_MIN*b[s, v] for (s, v) in BUILD[r]), name=f'sl_{r}')
    m.addConstrs((x[s, v, p] <= (LEGACY_CAP[s, r] if v == -1 else c[s, v])
                  for (s, v, p) in ACTIVE[r]), name=f'cap_{r}')
    m.addConstrs((gp.quicksum(ETA['MINE', v, p]*x['MINE', v, p]
                              for v in VIN[r, 'MINE', p]) == f_mp[p] for p in P),
                 name=f'mine_{r}')
    m.addConstrs((f_mp[p] == gp.quicksum(x['PROC', v, p] for v in VIN[r, 'PROC', p])
                  for p in P), name=f'pin_{r}')
    m.addConstrs((gp.quicksum(ETA['PROC', v, p]*x['PROC', v, p]
                              for v in VIN[r, 'PROC', p]) == f_pf[p] for p in P),
                 name=f'pout_{r}')
    m.addConstrs((f_pf[p] == gp.quicksum(x['MFG', v, p] for v in VIN[r, 'MFG', p])
                  for p in P), name=f'min_{r}')
    m.addConstrs((gp.quicksum(ETA['MFG', v, p]*x['MFG', v, p]
                              for v in VIN[r, 'MFG', p])
                  == sale.sum('*', p) + disp[p] for p in P), name=f'mout_{r}')

    # cumulative production (undiscounted), regional scope, with initial experience
    cum = m.addVars(P, lb=0.0, ub=3*CAP_MAX*HORIZON + EXPERIENCE0[r], name=f'cum_{r}')
    m.addConstrs((cum[p] == EXPERIENCE0[r] +
                  gp.quicksum(LEN[q]*x['MFG', v, q] for q in P if q <= p
                              for v in VIN[r, 'MFG', q]) for p in P), name=f'cp_{r}')

    capex = gp.quicksum(MU[s, v]*FIXED[s, r]*b[s, v] for (s, v) in BUILD[r]) \
          + gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                        for (s, v) in BUILD[r] if s not in LEARN_STAGES)
    if learning in ('capacity', 'both'):
        Q = m.addVars(P, lb=Q_START, ub=Q_START+Q_ADD, name=f'Q_{r}')
        Cc = m.addVars(P, lb=0.0, name=f'C_{r}')
        lam = m.addVars(P, K, lb=0.0, ub=1.0, name=f'lam_{r}')
        m.addConstrs((lam.sum(p, '*') == 1 for p in P), name=f'sc_{r}')
        m.addConstrs((Q[p] == gp.quicksum(QBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sQ_{r}')
        m.addConstrs((Cc[p] == gp.quicksum(CBP[k]*lam[p, k] for k in K) for p in P),
                     name=f'sC_{r}')
        m.addConstrs((Q[p] == Q_START + gp.quicksum(c[s, v] for (s, v) in BUILD[r]
                                                    if s in LEARN_STAGES and v <= p)
                      for p in P), name=f'cc_{r}')
        for p in P:
            m.addSOS(GRB.SOS_TYPE2, [lam[p, k] for k in K])
        rate = sum(UNIT[s, r] for s in LEARN_STAGES)/len(LEARN_STAGES)
        capex += gp.quicksum(MU['PROC', p]*rate*(Cc[p]-(Cc[p-1] if p > 0 else 0.0))
                             for p in P)
    else:
        capex += gp.quicksum(MU[s, v]*UNIT[s, r]*c[s, v]
                             for (s, v) in BUILD[r] if s in LEARN_STAGES)

    if learning in ('production', 'both') and TIER_Q:
        J = list(range(N_TIERS))
        z = m.addVars(P, J, vtype=GRB.BINARY, name=f'z_{r}')
        m.addConstrs((z.sum(p, '*') == 1 for p in P), name=f'ot_{r}')
        LAGP = {p: YEAR_TO_P[max(1, START[p]-LAG_YEARS)] for p in P}
        BIGQ = 3*CAP_MAX*HORIZON + EXPERIENCE0[r]
        m.addConstrs((cum[LAGP[p]] >= TIER_Q[r][j-1] - BIGQ*(1-z[p, j])
                      for p in P for j in J if j > 0), name=f'tf_{r}')
        m.addConstrs((cum[LAGP[p]] <= TIER_Q[r][j] + BIGQ*(1-z[p, j])
                      for p in P for j in J if j < N_TIERS-1), name=f'tc_{r}')
        ts = m.addVars(STAGES, P, J, lb=0.0, name=f'ts_{r}')
        m.addConstrs((ts.sum(s, p, '*') == gp.quicksum(x[s, v, p] for v in VIN[r, s, p])
                      for s in STAGES for p in P), name=f'tss_{r}')
        m.addConstrs((ts[s, p, j] <= 3*CAP_MAX*z[p, j]
                      for s in STAGES for p in P for j in J), name=f'tl_{r}')
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*TIER_M[r][j]*ts[s, p, j]
                           for s in STAGES for p in P for j in J)
    else:
        z = None
        opex = gp.quicksum(OMEGA[p]*OPEX[s, r]*x[s, v, p] for (s, v, p) in ACTIVE[r])

    trans = gp.quicksum(OMEGA[p]*TRANSPORT[r, rt]*sale[rt, p] for rt in REGIONS for p in P)
    dcost = gp.quicksum(OMEGA[p]*PEN_DISPOSE*disp[p] for p in P)
    revenue = gp.quicksum(OMEGA[p]*PRICE_FIXED*sale[rt, p] for rt in REGIONS for p in P)
    return dict(b=b, c=c, x=x, sale=sale, disp=disp, cum=cum, z=z,
                capex=capex, opex=opex, trans=trans, dcost=dcost, revenue=revenue,
                cost=capex+opex+trans+dcost)

In [ ]:
def solve_planner(w1=0.5, learning='both', mipgap=0.005, quiet=True):
    m = gp.Model(); m.Params.OutputFlag = 0 if quiet else 1; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    short = m.addVars(REGIONS, P, lb=0.0, name='short')
    m.addConstrs((gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS) + short[rt, p]
                  >= DEMAND[rt, p] for rt in REGIONS for p in P), name='demand')
    pen = gp.quicksum(OMEGA[p]*PEN_SHORT*short[rt, p] for rt in REGIONS for p in P)
    m.setObjective(w1*H['R1']['cost'] + (1-w1)*H['R2']['cost'] + pen, GRB.MINIMIZE)
    m.optimize()
    m._H, m._short, m._pen = H, short, pen
    return m

## 2. Inverse demand

$A$ is the choke price; $B$ is calibrated so that when quantity equals the Part 4b demand
reference, price equals `P_ANCHOR = 13` — the level at which the fixed-price game was interesting.
That makes 4b and 4c comparable at a reference point.

In [ ]:
# ================= 4c: Cournot with endogenous price =================
CHOKE    = 30.0     # price at zero quantity
P_ANCHOR = 13.0     # price when quantity equals the Part 4b demand reference
A_INT = {(rt, p): CHOKE for rt in REGIONS for p in P}
B_SLP = {(rt, p): (CHOKE - P_ANCHOR) / DEMAND[rt, p] for rt in REGIONS for p in P}


NBP_REV = 7          # breakpoints for the piecewise-linear revenue curve


def _rev_breakpoints(a_eff, b, smax, n=NBP_REV):
    """Breakpoints for revenue(s) = a_eff*s - b*s^2 on [0, smax].
    The function is CONCAVE and we MAXIMISE, so the chord between any two
    breakpoints lies BELOW the curve. A free convex combination therefore has no
    incentive to mix non-adjacent points -- unlike the concave-MINIMISE case in
    Part 3, this needs no SOS2 and adds no binaries."""
    S = [smax * k / (n - 1) for k in range(n)]
    R = [a_eff * x - b * x * x for x in S]
    return S, R

In [ ]:
def best_response_cournot(r, rival_sales, learning='both', mipgap=0.005):
    """Firm r maximises profit facing linear inverse demand
       p[rt,p] = A - B*(own + rival).
    Revenue is piecewise-linearised in own quantity, keeping the model a MILP."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    h = add_region(m, r, learning)
    s = h['sale']
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            q_bar = rival_sales.get((rt, p), 0.0)
            a_eff = A_INT[rt, p] - B_SLP[rt, p] * q_bar
            smax = max(1e-6, A_INT[rt, p] / B_SLP[rt, p] - q_bar)
            S, R = _rev_breakpoints(a_eff, B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1, name=f'rcvx_{rt}_{p}')
            m.addConstr(s[rt, p] == gp.quicksum(S[k] * mu[rt, p, k] for k in KR),
                        name=f'rS_{rt}_{p}')
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR),
                        name=f'rR_{rt}_{p}')
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - h['cost'], GRB.MAXIMIZE)
    m.optimize()
    m._h, m._rev = h, revenue
    return m

## 3. Convergence testing with continuous strategies

Part 4b tested convergence on the **build plan** alone. That is wrong here, and the failure mode is
instructive.

In Cournot the strategy *is* the quantity schedule. Testing plans alone declared convergence while
quantities were still moving by thousands of cost units. Fixing that by hashing the exact quantity
vector then produced a spurious **5-cycle** — profits oscillating in the fourth significant figure
with identical build plans. That was **MIP-gap noise**: each best response is a MILP solved to a
finite tolerance, so the returned quantities wobble slightly even at a genuine fixed point.

The correct test is **tolerance-based**, on the full strategy:

- **Converged** — the quantity profile moved less than `tol` since the previous round
- **Cycle** — the profile matches one from $k \ge 2$ rounds back, within `tol`
- **Cap** — report non-convergence

Two lessons that generalise: never hash floating-point strategies for equilibrium detection, and a
loose MIP gap inside a best-response loop can masquerade as strategic cycling.

In [ ]:
def cournot_iterate(learning='both', first='R1', max_iter=16, tol=0.5, mipgap=1e-3):
    """Iterated best response under Cournot competition.

    Convergence for a game with CONTINUOUS strategies must be tested with a
    TOLERANCE, not by exact state matching: each best response is a MILP solved to
    a finite gap, so the returned quantities wobble slightly between iterations.
    Exact hashing reads that wobble as a cycle."""
    def dist(a, b):
        return max(abs(a[r][k] - b[r][k]) for r in REGIONS for k in a[r])

    sales = {r: {(rt, p): 0.0 for rt in REGIONS for p in P} for r in REGIONS}
    plans, hist, log = {}, [], []
    order = [first, 'R2' if first == 'R1' else 'R1']
    for it in range(max_iter):
        prev = {r: dict(sales[r]) for r in REGIONS}
        for r in order:
            other = 'R2' if r == 'R1' else 'R1'
            m = best_response_cournot(r, sales[other], learning=learning, mipgap=mipgap)
            if m.SolCount == 0:
                return dict(status='INFEASIBLE', iters=it, log=log)
            sales[r] = {(rt, p): m._h['sale'][rt, p].X for rt in REGIONS for p in P}
            plans[r] = tuple(sorted((s_, v) for (s_, v) in m._h['b']
                                    if m._h['b'][s_, v].X > 0.5))
            log.append(dict(iter=it, firm=r, profit=m.ObjVal,
                            revenue=m._rev.getValue(), cost=m._h['cost'].getValue(),
                            builds=len(plans[r]), sales=sum(sales[r].values()),
                            disposal=sum(m._h['disp'][p].X for p in P)))
        cur = {r: dict(sales[r]) for r in REGIONS}
        if it > 0 and dist(cur, prev) < tol:
            return dict(status='CONVERGED', cycle_len=1, iters=it + 1, log=log,
                        plans=plans, sales=sales, drift=dist(cur, prev))
        for k, past in enumerate(hist):                      # genuine k-cycle, k >= 2
            if dist(cur, past) < tol:
                return dict(status='CYCLE', cycle_len=len(hist) - k, iters=it + 1,
                            log=log, plans=plans, sales=sales)
        hist.append(cur)
    return dict(status='MAX_ITER', iters=max_iter, log=log, plans=plans, sales=sales)

In [ ]:
def market_outcome(sales):
    rows = []
    for rt in REGIONS:
        for p in P:
            q = sum(sales[r][rt, p] for r in REGIONS)
            price = A_INT[rt, p] - B_SLP[rt, p] * q
            rows.append(dict(market=rt, period=p, year=START[p], quantity=q, price=price,
                             consumer_surplus=0.5 * B_SLP[rt, p] * q * q,
                             share_R1=(sales['R1'][rt, p] / q if q > 1e-6 else None)))
    return rows

In [ ]:
def joint_profit_max(learning='both', mipgap=0.005):
    """Collusive benchmark: one decision maker maximising the SUM of both profits."""
    m = gp.Model(); m.Params.OutputFlag = 0; m.Params.MIPGap = mipgap
    H = {r: add_region(m, r, learning) for r in REGIONS}
    KR = list(range(NBP_REV))
    mu = m.addVars(REGIONS, P, KR, lb=0.0, ub=1.0, name='mu')
    rev_t = m.addVars(REGIONS, P, lb=-GRB.INFINITY, name='revt')
    for rt in REGIONS:
        for p in P:
            smax = A_INT[rt, p] / B_SLP[rt, p]
            S, R = _rev_breakpoints(A_INT[rt, p], B_SLP[rt, p], smax, len(KR))
            m.addConstr(mu.sum(rt, p, '*') == 1)
            m.addConstr(gp.quicksum(H[r]['sale'][rt, p] for r in REGIONS)
                        == gp.quicksum(S[k] * mu[rt, p, k] for k in KR))
            m.addConstr(rev_t[rt, p] == gp.quicksum(R[k] * mu[rt, p, k] for k in KR))
    revenue = gp.quicksum(OMEGA[p] * rev_t[rt, p] for rt in REGIONS for p in P)
    m.setObjective(revenue - gp.quicksum(H[r]['cost'] for r in REGIONS), GRB.MAXIMIZE)
    m.optimize()
    m._H, m._rev = H, revenue
    return m

## 4. Calibrate the opex tiers, then solve the game

In [ ]:
m0 = solve_planner(0.5, learning='capacity')
top = {r: m0._H[r]['cum'][P[-1]].X for r in REGIONS}
set_tiers(top)
print("tier thresholds :", {r: [round(q, 1) for q in TIER_Q[r]] for r in REGIONS})
print("tier multipliers:", {r: [round(x, 3) for x in TIER_M[r]] for r in REGIONS})
print(f"\ninverse demand: A = {CHOKE}, price = {P_ANCHOR} at the 4b reference quantity")

In [ ]:
rows = []
for first in ['R1', 'R2']:
    res = cournot_iterate(first=first, max_iter=16)
    last = {L['firm']: L for L in res['log'][-2:]}
    rows.append(dict(first_mover=first, status=res['status'], iterations=res['iters'],
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     sales_R1=round(last['R1']['sales'], 1),
                     sales_R2=round(last['R2']['sales'], 1)))
pd.DataFrame(rows)

**Endogenous price largely dissolves the first-mover advantage.** In 4b, moving first was worth
about 29% of profit to R1 (7,613 vs 5,894). Here the two orders land within roughly 4% of each other.

The mechanism: under a fixed price the leader could commit capacity and *seize* a capped market,
leaving only leftovers. With a responsive price there is no cap to seize — if the leader floods, the
price falls and its own margin falls with it. Price adjustment substitutes for the quantity
rationing that gave commitment its bite.

This is worth taking seriously as a modelling lesson rather than a curiosity: **the large
first-mover advantage in 4b was substantially an artefact of the fixed price**, not a robust
finding. Whenever a result depends on a rationing rule, check what happens when a price does the
rationing instead.

## 5. Cournot against collusion

The natural benchmark now is not a cost-minimising planner but **joint profit maximisation** — the
two firms colluding as a single monopolist. Textbook prediction: collusion restricts quantity,
raises price, and earns higher joint profit.

In [ ]:
res = cournot_iterate(first='R1', max_iter=16)
jm  = joint_profit_max()
mo  = pd.DataFrame(market_outcome(res['sales']))
jsales = {r: {(rt, p): jm._H[r]['sale'][rt, p].X for rt in REGIONS for p in P}
          for r in REGIONS}
mj = pd.DataFrame(market_outcome(jsales))
cournot_joint = sum(L['profit'] for L in res['log'][-2:])
pd.DataFrame([
    dict(regime='Cournot duopoly', total_quantity=round(mo.quantity.sum(), 1),
         avg_price=round(mo.price.mean(), 2), joint_profit=round(cournot_joint, 1),
         consumer_surplus=round(mo.consumer_surplus.sum(), 1)),
    dict(regime='Collusion (joint max)', total_quantity=round(mj.quantity.sum(), 1),
         avg_price=round(mj.price.mean(), 2), joint_profit=round(jm.ObjVal, 1),
         consumer_surplus=round(mj.consumer_surplus.sum(), 1)),
])

Exactly the expected pattern, which is a useful validation that the price layer is wired correctly:

- **Collusion restricts output** (1,572 vs 2,238, about 30% less)
- **Price rises** (20.21 vs 16.05)
- **Joint profit rises** (22,870 vs 18,240) — competition costs the firms roughly 20% of profit
- **Consumers lose** under collusion

The Cournot equilibrium sits between monopoly and the competitive ideal, as it should.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for mk, col in zip(REGIONS, ['#2471a3', '#d68910']):
    d = mo[mo.market == mk]
    ax[0].plot(d.year, d.price, 'o-', lw=2.4, color=col, label=f'{mk} Cournot')
    dj = mj[mj.market == mk]
    ax[0].plot(dj.year, dj.price, 's--', lw=2.0, color=col, alpha=0.6,
               label=f'{mk} collusion')
ax[0].set_xlabel('year'); ax[0].set_ylabel('price'); ax[0].legend(fontsize=9)
ax[0].set_title('Collusion holds price above Cournot')
for mk, col in zip(REGIONS, ['#2471a3', '#d68910']):
    d = mo[mo.market == mk]
    ax[1].plot(d.year, d.share_R1, 'o-', lw=2.4, color=col, label=f'market {mk}')
ax[1].axhline(0.5, ls=':', color='k')
ax[1].set_xlabel('year'); ax[1].set_ylabel("R1's share of the market")
ax[1].set_ylim(0, 1); ax[1].legend(fontsize=10)
ax[1].set_title("Incumbent's share: home market vs entrant's market")
plt.tight_layout(); plt.show()

The share panel shows the asymmetry doing its work. R1 holds about 65% of its **home** market, where
it has both the transport advantage and its accumulated experience, but only about 49% of R2's
market, where the 2.4 cross-region transport premium offsets its lower operating cost. Geography and
experience push in different directions, and the equilibrium splits the difference market by market.

## 6. Does learning drive output? The flooding channel

This is the question Part 3b could not answer. There, cumulative production was pinned by demand,
so production learning was a windfall that changed cost but not a single decision. Here quantity is
a genuine decision, so the channel finally has a lever.

In [ ]:
rows = []
for lm in ['capacity', 'both']:
    r2 = cournot_iterate(learning=lm, first='R1', max_iter=16)
    last = {L['firm']: L for L in r2['log'][-2:]}
    m2 = pd.DataFrame(market_outcome(r2['sales']))
    rows.append(dict(learning=lm, status=r2['status'],
                     total_quantity=round(m2.quantity.sum(), 1),
                     avg_price=round(m2.price.mean(), 2),
                     sales_R1=round(last['R1']['sales'], 1),
                     sales_R2=round(last['R2']['sales'], 1),
                     profit_R1=round(last['R1']['profit'], 1),
                     profit_R2=round(last['R2']['profit'], 1),
                     disposal=round(last['R1']['disposal'] + last['R2']['disposal'], 2)))
pd.DataFrame(rows)

**Adding the production-learning channel raises total output by about 13%** (1,976 → 2,238) and
pushes average price down from 17.65 to 16.05. That is the flooding mechanism, and it is the first
time in this whole series that a learning channel has changed *quantities* rather than only costs.

The logic: a unit sold today is worth more than its immediate margin, because it advances cumulative
production toward a cheaper operating-cost tier. Firms therefore rationally sell **past** the
static profit-maximising quantity — which depresses price for both of them. Learning makes the
market more competitive.

Note also that R1 gains more output than R2 (1,061 → 1,267 versus 916 → 971). The incumbent starts
closer to the next tier, so its marginal unit buys more learning. **Learning-by-doing amplifies
incumbency** rather than helping the entrant catch up — a result that matters for industrial policy
and that only appears once quantity is endogenous.

**Disposal stays at exactly zero.** Firms flood by *selling* at a depressed price, not by dumping:
disposal destroys the revenue while still paying the production cost, so it is dominated. This
confirms the Part 3b calibration — `PEN_DISPOSE = 12` is comfortably above any threshold where
dumping would appear, and the mechanism remains correctly wired and idle.

## 7. Summary

| Question | Answer |
|---|---|
| Does endogenous price change the 4b conclusions? | **Yes** — first-mover advantage falls from ~29% to ~4% |
| Cournot vs collusion? | Collusion cuts output ~30%, raises price 16.05 → 20.21, joint profit +20% |
| Does production learning drive output now? | **Yes** — +13% quantity, price down 1.60 |
| Who benefits from learning? | **The incumbent** — it is closer to the next tier |
| Does pump-and-dump appear? | No. Flooding happens through **sales**, not disposal |
| Pure-strategy equilibrium? | Yes, from both move orders |

### Formulation lessons

- **Curvature and direction together decide whether you need SOS2.** Concave-and-maximised revenue
  is safe with free $\lambda$; concave-and-minimised cost (Part 3) is not. Same shape, opposite
  requirement.
- **Piecewise-linearising revenue avoided a MIQP entirely** — and kept the model inside the
  size-limited licence.
- **Test convergence on the actual strategy, with a tolerance.** Plans alone declared premature
  convergence; exact quantity hashing invented a 5-cycle out of MIP-gap noise.
- **Results that depend on a rationing rule deserve suspicion.** 4b's first-mover advantage was
  mostly an artefact of the fixed price.

### Still to build

- **4d — Stackelberg.** Leader commits, follower responds. Because the follower's *operational*
  problem is an LP, its KKT conditions can be written explicitly and the bilevel model collapsed to
  a single-level MPEC with big-M complementarity. This is the payoff for keeping flows continuous
  all the way back in Part 3.
- **4e — Policy instruments.** Tariffs as arc-cost adders, quotas as arc bounds, local content
  minimums (already built in Part 3b), all exogenous and swept. Government constrains, firms
  respond.

### Things to try

- `CHOKE = 24` — a flatter market; competition bites harder and margins compress
- `P_ANCHOR = 10` — barely profitable, and capacity investment should collapse
- `EXPERIENCE0 = {'R1': 500, 'R2': 500}` — remove incumbency and watch the learning amplification
  disappear
- `TRANSPORT` cross-region down to 1.0 — one integrated market instead of two linked ones
- `NBP_REV = 3` — a coarse revenue mesh; quantities should get visibly blocky